# 01 — Entender imagens, lesões e limitações

## Objetivo
Quantificar a unidade observada e a unidade independente. **Número de imagens ≠ número de lesões independentes.** Não preparar dados nem escolher modelos aqui.

## Entradas
RAW, `GroundTruth.csv`, suplemento oficial de lesões e auditoria v2 já disponível.

## Saídas
Tabelas e figuras de EDA nesta execução. Nenhum CSV de treinamento é alterado.

## Permissões de dados
Metadados globais para inventário; exemplos visuais somente das imagens atribuídas a train v2. Sem métricas ou inspeção de erros de modelos.

## Reprodutibilidade
Amostras visuais determinísticas: seed 42. Contagens completas; proveniência do suplemento verificada.

Protocolo: [experimental_protocol_v2.md](../docs/experimental_protocol_v2.md).
Os resultados anteriores estão catalogados como `legacy_v1`; não comparar seus scores diretamente aos v2.


In [ ]:
from pathlib import Path
import sys

# Bootstrap de importação; a resolução dos dados fica nos módulos compartilhados.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "scripts/experimental_protocol.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from scripts.v2_artifacts import repository_root, read_json, run_id
ROOT = repository_root()

from scripts.experimental_validity import CLASSES, official_metadata, sha256_file
from scripts.v2_data import METADATA_REL, AUDIT_REL, SPLIT_REL, read_audit
SEED = 42
metadata = official_metadata(ROOT, ROOT / METADATA_REL)
inventory, audit_summary = read_audit(ROOT)
assert sha256_file(ROOT / METADATA_REL) == audit_summary["source_hashes"][METADATA_REL]
assert set(metadata.image_stem) == set(inventory.image_stem)
display(pd.Series({"dataset": "HAM10000 / ISIC 2018 Task 3", "imagens": len(metadata),
                   "lesões": metadata.lesion_id.nunique(), "patient_id disponível": False,
                   "metadata_sha256": sha256_file(ROOT / METADATA_REL)}).to_frame("valor"))


## Origem e limites da identidade
O arquivo local `data/raw/lesions/images/ATTRIBUTION.txt` identifica HAM10000 / ISIC 2018 Task 3.
O suplemento oficial recuperado na fase de validade contém `image`, `lesion_id` e `diagnosis_confirm_type`.
Não contém `patient_id`: separação por lesão não estabelece independência por paciente.
Método de confirmação não identifica paciente, centro ou qualidade clínica da imagem.


In [ ]:
counts = pd.DataFrame({"imagens": metadata.target_label.value_counts(),
                       "lesões": metadata.groupby("target_label").lesion_id.nunique()}).reindex(CLASSES)
counts["imagens_por_lesão"] = counts.imagens / counts.lesões
display(counts)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, unit in zip(axes, ["imagens", "lesões"]):
    counts[unit].plot.bar(ax=ax, color="steelblue")
    ax.set(title=f"Distribuição das sete classes — {unit}", xlabel="Classe", ylabel=f"Número de {unit}")
    ax.tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()


## Dependência entre vistas
Classes com mais imagens não necessariamente têm proporcionalmente mais lesões.
Mais vistas de uma lesão ajudam a representar variação de captura, mas não criam novos pacientes independentes.


In [ ]:
views = metadata.groupby("lesion_id").size()
distribution = views.value_counts().sort_index()
display(distribution.rename_axis("imagens_por_lesão").to_frame("número_de_lesões"))
ax = distribution.plot.bar(figsize=(8, 4), color="teal")
ax.set(title="Quantas vistas existem por lesão?", xlabel="Imagens da mesma lesão", ylabel="Número de lesões")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout(); plt.show()
display(inventory.groupby(["width", "height", "mask_width", "mask_height"]).size().to_frame("imagens"))
display(metadata.diagnosis_confirm_type.value_counts().to_frame("imagens por método de confirmação"))


## Duplicatas e correspondência imagem/máscara
Hash de arquivo confirma cópias exatas. pHash apenas identifica candidatos para revisão: proximidade não prova identidade.
Máscaras têm origem local não documentada e não são utilizadas como feature de classificação.
O overlay abaixo mostra correspondência espacial, não uma validação clínica da anotação.


In [ ]:
display(pd.Series(audit_summary["pair_categories"]).to_frame("pares auditados"))
display(inventory.loc[inventory.mask_foreground_fraction.isin([0, 1]),
                      ["image_stem", "mask_foreground_fraction"]])
split_manifest = read_json(ROOT / SPLIT_REL / "split_manifest.json")
train_path = ROOT / SPLIT_REL / "train.csv"
assert sha256_file(train_path) == split_manifest["artifact_hashes"]["train.csv"]
examples = pd.read_csv(train_path, keep_default_na=False)
from PIL import Image, ImageOps
row = examples.sample(1, random_state=SEED).iloc[0]
with Image.open(ROOT / row.image_path) as im:
    rgb = ImageOps.exif_transpose(im).convert("RGB").copy()
with Image.open(ROOT / row.mask_path) as im:
    mask = im.copy()
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(rgb); axes[0].set_title(f"Train: {row.image_stem} / {row.target_label}")
axes[1].imshow(mask, cmap="gray"); axes[1].set_title("Máscara correspondente")
axes[2].imshow(rgb); axes[2].imshow(mask, alpha=.3, cmap="Reds"); axes[2].set_title("Correspondência espacial")
for ax in axes: ax.axis("off")
fig.tight_layout(); plt.show()


## Múltiplas imagens da mesma lesão
A escolha abaixo usa somente a quantidade de vistas e IDs ordenados. Não usa confiança, erro ou predições.


In [ ]:
lesion = examples.groupby("lesion_id").size().sort_values(ascending=False, kind="stable").index[0]
same_lesion = examples.loc[examples.lesion_id == lesion].sort_values("image_stem")
fig, axes = plt.subplots(1, len(same_lesion), figsize=(3 * len(same_lesion), 3), squeeze=False)
for ax, row in zip(axes.flat, same_lesion.itertuples()):
    with Image.open(ROOT / row.image_path) as im:
        ax.imshow(ImageOps.exif_transpose(im).convert("RGB"))
    ax.set_title(row.image_stem); ax.axis("off")
fig.suptitle(f"Uma lesão ({lesion}), {len(same_lesion)} imagens de train")
fig.tight_layout(); plt.show()


## Conclusão
As contagens acima descrevem imagens e lesões separadamente. O corpus é desbalanceado, não possui patient_id e já foi observado em desenvolvimento. A EDA não estabelece desempenho de generalização.

## Artefatos produzidos
Tabelas e figuras no notebook. A auditoria persistida continua sendo a fonte de hashes, sem regeneração nesta etapa.

## Próxima etapa
NB02: verificar RAW, auditoria e todas as invariantes antes de emitir o certificado do split.
